# DAAN 545 — Outliers and the clustering fit

**Goal:** See whether the five delay patterns change if outlier months are removed before K-Means and Ward.

**Input:** `data/processed/airline_delay_cause_with_airports.csv`. Same months and same six features as `notebooks/03_clustering.ipynb`. FAA staffing is not an input.

**Approach:**
1. Fit the published clustering, which **retains** every scored month.
2. Refit after three removals. Each refit re-standardizes the rows it keeps and chooses k from the K-Means silhouette (k = 2…8).
3. Match the new groups back to the retained groups and compare cancel rates, weather share, and membership.

Write-up language and the numbers from the saved run: `docs/notebook_notes.md`.


## 0. Setup

Helpers live in `src/cluster_outlier_impact.py`. Outlier flags come from `src/outliers.py`. The flight-count cutoff is the one already used for clustering. It is not recomputed after a removal.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cluster_outlier_impact import run_outlier_impact, save_outlier_impact_figures

DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
print(f"Project root: {ROOT}")


## 1. What is removed

| Scenario | Months dropped |
|---|---|
| **retain** | None. This is the published fit. |
| **drop consensus** | Flagged by at least two of IQR rates, k-NN, and isolation forest. |
| **drop delay IQR** | `pct_delayed` outside the Tukey fence only. |
| **drop any flag** | Flagged by any one of the three outlier methods. |

A new cluster is given the retained role (late aircraft, NAS, carrier, cancellation, weather) whose mean profile is closest after each mean is scaled by the spread of the retained profiles. Cluster ids themselves are not comparable across fits.


In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "airline_delay_cause_with_airports.csv")
result = run_outlier_impact(panel)

print("Scored months:", result["meta"]["n_rows"])
print("Features:", result["meta"]["features"])
display(result["summary"])


## 2. Profiles and membership

`same_role_pct` is the share of **kept** months whose new group matches the retained group they started in. The profile table is the refit, not the retained rows only.


In [ ]:
stability = result["stability"]
display(stability.loc[stability["role"] == "all"])
print("K-Means, by retained role:")
display(stability.loc[stability["method"] == "kmeans"])

keep_cols = [
    "scenario",
    "method",
    "role",
    "n_rows",
    "mean_pct_delayed",
    "mean_pct_cancelled",
    "mean_share_carrier",
    "mean_share_weather",
    "mean_share_nas",
    "mean_share_late_aircraft",
    "top_season",
]
display(result["profiles"][keep_cols])


## 3. Figures and exports

The silhouette curves use the same k grid as the published clustering. The bars are K-Means means for the retained fit, the consensus removal, and the removal of every flag.


In [ ]:
paths = save_outlier_impact_figures(result, FIG_DIR)
for path in paths:
    print(path)

result["summary"].to_csv(DATA_PROCESSED / "cluster_outlier_scenario_summary.csv", index=False)
result["profiles"].to_csv(DATA_PROCESSED / "cluster_outlier_profiles.csv", index=False)
result["stability"].to_csv(DATA_PROCESSED / "cluster_outlier_stability.csv", index=False)
result["sweeps"].to_csv(DATA_PROCESSED / "cluster_outlier_k_sweep.csv", index=False)
result["pattern_by_cluster"].to_csv(DATA_PROCESSED / "cluster_outlier_pattern_counts.csv", index=False)
print("Saved outlier-impact tables under data/processed/")
